# Train PatchCore on Google Colab

Trains the defect detector on a free Colab GPU and gives you `<category>_model.zip`. Put that zip in the
project root on your computer, then run `inference.py`. It extracts the zip into `models/` automatically.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

In [ ]:
!nvidia-smi -L || echo "No GPU: training will work but coreset selection is much slower."

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/Rafey28/defect-detection.git"

%cd /content
!git clone -q {REPO_URL} defect-detection || git -C defect-detection pull -q
%cd /content/defect-detection
!pip install -q -r requirements.txt

## 2. Provide the dataset

The MVTec AD dataset is not in the repo. Pick one option:

* **A. Google Drive (recommended, survives restarts):** make `data.zip` on your computer with `python dataset_zip.py`,
  upload it to `MyDrive/industrial_defect_detection/data.zip`, then run the next cell. The trained model is also saved there.
* **B. Direct upload:** skip the next cell and drag `data.zip` into `/content/defect-detection/` in the Files panel.
  Uploads and results are deleted when the session ends.

`train.py` looks for `data/mvtec/<category>/` first, then `data.zip` in the project folder, then on Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Train
Other categories: `screw`, `tile`, … (any MVTec AD category in your data).

In [ ]:
CATEGORY = "metal_nut"
!python train.py --category {CATEGORY}

## 4. (Optional) Check one prediction here

In [ ]:
import glob
from IPython.display import Image, display

defect_images = sorted(p for p in glob.glob(f"data/**/{CATEGORY}/test/*/000.png", recursive=True) if "/good/" not in p)
sample = defect_images[0]
!python inference.py --category {CATEGORY} --trust-model --image "{sample}" --output-dir output_inference
display(Image(sorted(glob.glob("output_inference/*.png"))[-1], width=700))

## 5. Get the model onto your computer
If Drive is mounted, the zip is already at `MyDrive/industrial_defect_detection/models/<category>_model.zip`; download it from Drive.
Otherwise, run the cell below. Then put the zip in the project root and run:

```
python inference.py --category metal_nut --trust-model --image ./data/mvtec/metal_nut/test/scratch/000.png
```

In [ ]:
from google.colab import files
files.download(f"{CATEGORY}_model.zip")